In [52]:
import os
from utils_extraction import extract_body, tokenize, clean_tokens, decode
from utils_extraction import chunk_tokens, flatten_token_chunks
from utils_extraction import extract_few_shot_examples
from utils_extraction import select_few_shot 
from utils_extraction import merge_tokens_with_auto_labels, add_attributes_to_auto_labels, compare_html_allow_auto_labels, correct_tokens_brackets, check_tokens_brackets
from models import GPTAssistant
from utils_extraction import process_chunks
from utils_extraction.html_utils import clean_html_formatting

In [75]:
# ---------- Define Hyperparameters ----------
min_tokens = 500
fs_min_tokens = 200
model_name = "gpt-5.2"

n_few_shot = 15  # Number of few-shot examples to use

#### Define the text to process, and where to save it. Define the text for few shot

In [76]:
# File paths
project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
filename = "2021QCCA1675"
round = "ronde_2"
anno = "llm"
version = "v1.0"
html_path = fr"{project_root}\data\Document_Échantillon_Initial\{round}\plain_html_arbre_balise\{filename}.html"
output_dir = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}\v_prompt_2_500_100_15_gpt5.2"

os.makedirs(output_dir, exist_ok=True)
# Read HTML file
with open(html_path, 'r', encoding='utf-8') as file:
    html_content = file.read()
print(f"   ✓ HTML file loaded: {html_path}")


fs_filename = "2019SCC65_annotated_EG_v1_corrected"
fs_anno = "EG"
fs_version = "v1"
fs_html_path = fr"{project_root}\data\Documents_Annotés\{fs_anno}\{fs_filename}.html"
# Read HTML file
with open(fs_html_path, 'r', encoding='utf-8') as file:
    fs_html_content = file.read()
print(f"   ✓ HTML file loaded for few shot: {fs_html_path}")


   ✓ HTML file loaded: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Document_Échantillon_Initial\ronde_2\plain_html_arbre_balise\2021QCCA1675.html
   ✓ HTML file loaded for few shot: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\EG\2019SCC65_annotated_EG_v1_corrected.html


### Process The HTML Content

In [77]:

# ---------- Extract body content ----------
body_content = extract_body(html_content)


# ---------- Tokenize body content ----------
tokens = tokenize(body_content)

# ---------- Clean tokens ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
token_chunks = chunk_tokens(normalized_cleaned_tokens, min_tokens=fs_min_tokens, stop_bookmark_separation=True)


   ⚠ Warning: stop_bookmark_separation=True but bookmark not found
   ✓ Chunked tokens into 46 chunks (>= 200 tokens each)


In [78]:
# ---------- Extract body content ----------
fs_body_content = extract_body(fs_html_content)


# ---------- Tokenize body content ----------
fs_tokens = tokenize(fs_body_content)

# ---------- Clean tokens ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=fs_tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
token_chunk1, token_chunk2 = chunk_tokens(normalized_cleaned_tokens, min_tokens=min_tokens, stop_bookmark_separation=True)

   ✓ Found bookmark separator at index 43561
   ✓ Splitting: 43561 tokens before, 102341 tokens after
   ✓ Chunked tokens into 62 chunks (>= 500 tokens each)
   ✓ Chunked tokens into 204 chunks (>= 500 tokens each)
   ✓ Total chunks: 62 before + 204 after = 266


In [79]:
label_config = {
    "keep_attributes":["labelname"], # extraction only, no disambiguation
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    "keep_labels":["decision", "legislation", "secondary sources"]
}

In [80]:
# ---------- Create few-shot examples ----------

few_shot_examples = extract_few_shot_examples(token_chunk1, 
                                              label_config)



selected_few_shot_examples = select_few_shot(examples=few_shot_examples, n=n_few_shot, 
                                             method="distributed", 
                                             list_of_labels=["decision", "legislation", "secondary sources"], 
                                             distribution=[0.3, 0.3, 0.3])
print(f"   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")

   ✓ Extracted 62 few-shot examples from chunks
   ⚠ Warning: Requested 4 examples with label 'secondary sources', but only 2 available
   ✓ Selected 15 few-shot examples for processing.


In [81]:
for examples in selected_few_shot_examples:
    print("==============================================================================")
    print("input: \n ", examples[0])
    print("------------------------------------------------------------------------------")
    print("output: \n ", examples[1])

input: 
  ] Reasonableness review is an approach meant to ensure that courts intervene in administrative matters only where it is truly necessary to do so in order to safeguard the legality, rationality and fairness of the administrative process. It finds its starting point in the principle of judicial restraint and demonstrates a respect for the distinct role of administrative decision makers. However, it is not a “rubber-stamping” process or a means of sheltering administrative decision makers from accountability. It remains a robust form of review.
[14] On the one hand, courts must recognize the legitimacy and authority of administrative decision makers within their proper spheres and adopt an appropriate posture of respect. On the other hand, administrative decision makers must adopt a culture of justification and demonstrate that their exercise of delegated public power can be “justified to citizens in terms of rationality and fairness”: the Rt. Hon. B. McLachlin, “The Roles of Ad

In [82]:
# ---------- Initialize LLM model ----------
model = GPTAssistant(model_name, temperature=1)

In [83]:
# ---------- Process chunks ----------

prompt_path = fr"{project_root}\llm_based_annotation\utils_extraction\prompts\simplified_parent_extraction_cot v2.txt"

processed_chunks = process_chunks(
    model=model,
    token_chunks=token_chunks,
    process_prompt_path=prompt_path,
    label_config=label_config,
    few_shot_examples=selected_few_shot_examples,
    output_dir=output_dir,
    filename=filename
)


   ✓ Processing 46 chunks with LLM...
   ✓ Using 15 few-shot examples


Processing chunks:  43%|████▎     | 20/46 [01:13<01:33,  3.59s/it]

   ⚠ Warning: <start> marker found but <end> marker missing


Processing chunks: 100%|██████████| 46/46 [02:53<00:00,  3.78s/it]

   ✓ Processing history saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2021QCCA1675\v_prompt_2_500_100_15_gpt5.2\history_2021QCCA1675.json

   ✓ Processing completed:
      - Total chunks: 46
      - Successful: 46
      - Failed: 0
   ✓ Processed chunks saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2021QCCA1675\v_prompt_2_500_100_15_gpt5.2\processed_chunks_2021QCCA1675.json


In [84]:
#write the processed chuncks in a json file for later use in the annotation interface

import json 
with open(f"{output_dir}\\processed_chunks.json", "w") as f:
    json.dump(processed_chunks, f)

## Post Processing

In [37]:
# Read the processed_chuncks.json file to verify it was written correctly
import json
with open(f"{output_dir}\\processed_chunks.json", "r") as f:
    processed_chunks = json.load(f)

In [85]:
# Processed_chunks is a list of lists of tokens, we need to flatten it to get a single list of tokens for the whole document
processed_tokens_flat = flatten_token_chunks(processed_chunks)


# Read in parallel the original tokens and the processed tokens. Always prefer the original tokens, but if there is an auto_label token in the processed tokens, 
# we want to keep it and merge it with the original tokens. 
# This way we can keep the original formatting and structure of the document while adding the auto_labels extracted by the model.
original_tokens = tokenize(html_content)
processed_html_content_tokens = merge_tokens_with_auto_labels(original_tokens, processed_tokens_flat)

# This merging process can sometimes create some formatting issues with brackets, we need to correct them to get a valid HTML structure.
processed_html_content_tokens_corrected = correct_tokens_brackets(processed_html_content_tokens)
assert check_tokens_brackets(processed_html_content_tokens_corrected), "The brackets in the merged tokens are not balanced. Please check the merging and bracket correction steps for errors."



# The correction of the brackets can sometimes create some redoundant or useless formatting  with the HTML, we need to clean it to compare it with the original.
processed_html = decode(processed_html_content_tokens_corrected)
processed_html_cleaned = clean_html_formatting(processed_html)
print(f"\nMerged HTML length: {len(processed_html_cleaned)}")


# This step is just to ensure a good visualisation of HTMLLabelizer and to add the necessary attribute to stay consistent with the label scheme
processed_html_content = add_attributes_to_auto_labels(processed_html_cleaned)


# Last check of the final processed_html_content with the original HTML, ignoring the auto_label tags which are not present in the original HTML but only in the processed one.
comparison_result = compare_html_allow_auto_labels(processed_html_content, html_content)
assert comparison_result, "The processed HTML content does not match the original HTML content when ignoring auto_label tags. Please check the merging and post-processing steps for errors."


# ---------- Save processed HTML to file ----------
with open(fr"{output_dir}\{filename}_llm_{version}.html", 'w', encoding='utf-8') as f:
    f.write(processed_html_content)
print(f"   ✓ Processed HTML saved to: {output_dir}")

   ✓ Flattened 46 chunks into 10752 tokens

Merged HTML length: 119782
   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)
   ✓ Processed HTML saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\2021QCCA1675\v_prompt_2_500_100_15_gpt5.2
